In [1]:
# Install required packages
!pip install nltk python-Levenshtein matplotlib torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.9 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
import json
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import nltk
import Levenshtein
from collections import Counter
import math
import random

# Download required NLTK data

nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)


True

In [3]:
def encode_sentence(sentence, token2id, is_urdu=True):
    """Encode a sentence using greedy longest-match subword tokenization."""
    tokens = []

    if is_urdu:
        words = ["_" + w for w in sentence.split()]
    else:
        words = [w + "_" for w in sentence.split()]

    for w in words:
        i = 0
        while i < len(w):
            subword = None
            for j in range(len(w), i, -1):
                piece = w[i:j]
                if piece in token2id:
                    subword = piece
                    break
            if subword is None:
                tokens.append(token2id["<unk>"])
                i += 1
            else:
                tokens.append(token2id[subword])
                i += len(subword)

    return [token2id["<sos>"]] + tokens + [token2id["<eos>"]]



class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab):
        self.src_data = []
        self.tgt_data = []

        for src, tgt in zip(src_sentences, tgt_sentences):
            src_tokens = encode_sentence(src, src_vocab, is_urdu=True)
            tgt_tokens = encode_sentence(tgt, tgt_vocab, is_urdu=False)

            self.src_data.append(torch.tensor(src_tokens))
            self.tgt_data.append(torch.tensor(tgt_tokens))

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        return self.src_data[idx], self.tgt_data[idx]


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)

    # lengths
    max_src_len = max(len(src) for src in src_batch)
    max_tgt_len = max(len(tgt) for tgt in tgt_batch)

    # we force both to same max length
    max_len = max(max_src_len, max_tgt_len)

    # pad each sequence with 0 up to max_len
    src_padded = [F.pad(src, (0, max_len - len(src)), value=0) for src in src_batch]
    tgt_padded = [F.pad(tgt, (0, max_len - len(tgt)), value=0) for tgt in tgt_batch]

    # stack into batch tensors
    return torch.stack(src_padded), torch.stack(tgt_padded)



In [4]:


class Encoder(nn.Module):
    def __init__(self, vocab_size=512, embed_dim=128, hidden_dim=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.bilstm = nn.LSTM(embed_dim, hidden_dim, num_layers,
                             dropout=dropout, bidirectional=True, batch_first=True)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

    def forward(self, x):
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)
        output, (h, c) = self.bilstm(embedded)
        return output, (h, c)

class Decoder(nn.Module):
    def __init__(self, vocab_size=512, embed_dim=256, hidden_dim=256, num_layers=4,
                 output_vocab_size=512, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers,
                           dropout=dropout, batch_first=True)
        self.linear = nn.Linear(hidden_dim, output_vocab_size)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        # Project encoder states to decoder dimensions
        self.h_projection = nn.Linear(256, hidden_dim)  # 256 from bidirectional encoder
        self.c_projection = nn.Linear(256, hidden_dim)

    def forward(self, x, encoder_states=None):
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)

        if encoder_states is not None:
            h_enc, c_enc = encoder_states
            # Convert bidirectional encoder states to decoder format
            # h_enc: [num_layers*2, batch, hidden_dim] -> [num_layers, batch, hidden_dim*2]
            batch_size = h_enc.size(1)
            h_enc = h_enc.view(2, 2, batch_size, -1)  # [directions, layers, batch, hidden]
            c_enc = c_enc.view(2, 2, batch_size, -1)

            # Concatenate forward and backward states
            h_enc = torch.cat([h_enc[0], h_enc[1]], dim=-1)  # [layers, batch, hidden*2]
            c_enc = torch.cat([c_enc[0], c_enc[1]], dim=-1)

            # Project to decoder dimensions and repeat for all decoder layers
            h_init = self.h_projection(h_enc[-1]).unsqueeze(0).repeat(self.num_layers, 1, 1)
            c_init = self.c_projection(c_enc[-1]).unsqueeze(0).repeat(self.num_layers, 1, 1)

            initial_state = (h_init, c_init)
        else:
            initial_state = None

        output, _ = self.lstm(embedded, initial_state)
        output = self.linear(output)
        return output

class Seq2SeqModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, src):
        # Encoder processes source
        enc_output, enc_states = self.encoder(src)

        # Decoder processes same source sequence (as per your requirement)
        dec_output = self.decoder(src, enc_states)

        return dec_output

In [5]:
def load_data_and_vocab():
    """Load data and vocabularies from Google Drive"""
    base_path = "/content/drive/MyDrive/Model2/"

    # Load source and target sentences
    with open(base_path + "src_normalized.txt", 'r', encoding='utf-8') as f:
        src_sentences = [line.strip() for line in f]

    with open(base_path + "tgt_normalized.txt", 'r', encoding='utf-8') as f:
        tgt_sentences = [line.strip() for line in f]

    # Load vocabularies
    with open(base_path + "vocab_Urdu.json", 'r', encoding='utf-8') as f:
        urdu_vocab = json.load(f)

    with open(base_path + "vocab_Roman.json", 'r', encoding='utf-8') as f:
        roman_vocab = json.load(f)

    return src_sentences, tgt_sentences, urdu_vocab, roman_vocab

def create_datasets(src_sentences, tgt_sentences, urdu_vocab, roman_vocab):
    """Create train/val/test splits"""
    total_size = len(src_sentences)
    train_size = int(0.7 * total_size)
    val_size = int(0.15 * total_size)

    # Shuffle data
    indices = list(range(total_size))
    random.shuffle(indices)

    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]

    # Create datasets
    train_src = [src_sentences[i] for i in train_indices]
    train_tgt = [tgt_sentences[i] for i in train_indices]

    val_src = [src_sentences[i] for i in val_indices]
    val_tgt = [tgt_sentences[i] for i in val_indices]

    test_src = [src_sentences[i] for i in test_indices]
    test_tgt = [tgt_sentences[i] for i in test_indices]

    train_dataset = TranslationDataset(train_src, train_tgt, urdu_vocab, roman_vocab)
    val_dataset = TranslationDataset(val_src, val_tgt, urdu_vocab, roman_vocab)
    test_dataset = TranslationDataset(test_src, test_tgt, urdu_vocab, roman_vocab)

    return train_dataset, val_dataset, test_dataset

def calculate_perplexity(loss):
    """Calculate perplexity from loss"""
    return math.exp(loss)

def decode_tokens(tokens, id2token):
    """Convert token IDs back to text"""
    words = []
    for token_id in tokens:
        if token_id in [0, 1, 2]:  # pad, sos, eos
            continue
        words.append(id2token.get(token_id, '<unk>'))
    return ' '.join(words)

def calculate_bleu(reference, hypothesis):
    """Calculate BLEU score"""
    reference_tokens = reference.split()
    hypothesis_tokens = hypothesis.split()

    if len(hypothesis_tokens) == 0:
        return 0.0

    smoothie = SmoothingFunction().method5
    return sentence_bleu([reference_tokens], hypothesis_tokens, smoothing_function=smoothie)

def calculate_cer(reference, hypothesis):
    """Calculate Character Error Rate"""
    if len(reference) == 0:
        return 1.0 if len(hypothesis) > 0 else 0.0
    return Levenshtein.distance(reference, hypothesis) / len(reference)

def calculate_edit_distance(reference, hypothesis):
    """Calculate Levenshtein distance"""
    return Levenshtein.distance(reference, hypothesis)


In [6]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device, roman_vocab):
    id2roman = {v: k for k, v in roman_vocab.items()}

    for epoch in range(epochs):
        # ---- TRAINING ----
        model.train()
        total_loss = 0
        correct, total = 0, 0

        for batch_idx, (src, tgt) in enumerate(train_loader):
            src, tgt = src.to(device), tgt.to(device)

            optimizer.zero_grad()
            output = model(src)  # (batch, seq_len, vocab_size)

            # Flatten for CE Loss
            output_flat = output.reshape(-1, output.size(-1))
            tgt_flat = tgt.reshape(-1)

            loss = criterion(output_flat, tgt_flat)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()

            # Accuracy (token-level)
            predictions = torch.argmax(output, dim=-1)
            correct += (predictions == tgt).sum().item()
            total += tgt.numel()

        avg_train_loss = total_loss / len(train_loader)
        train_perplexity = calculate_perplexity(avg_train_loss)
        train_accuracy = correct / total

        # ---- VALIDATION ----
        model.eval()
        val_loss, val_bleu, val_cer, val_edit, val_correct, val_total = 0, 0, 0, 0, 0, 0

        with torch.no_grad():
            for src, tgt in val_loader:
                src, tgt = src.to(device), tgt.to(device)
                output = model(src)

                # Loss
                output_flat = output.reshape(-1, output.size(-1))
                tgt_flat = tgt.reshape(-1)
                loss = criterion(output_flat, tgt_flat)
                val_loss += loss.item()

                # Predictions
                predictions = torch.argmax(output, dim=-1)
                val_correct += (predictions == tgt).sum().item()
                val_total += tgt.numel()

                for i in range(src.size(0)):
                    pred_tokens = predictions[i].cpu().numpy()
                    tgt_tokens = tgt[i].cpu().numpy()

                    pred_text = decode_tokens(pred_tokens, id2roman)
                    ref_text = decode_tokens(tgt_tokens, id2roman)

                    val_bleu += calculate_bleu(ref_text, pred_text)
                    val_cer += calculate_cer(ref_text, pred_text)
                    val_edit += calculate_edit_distance(ref_text, pred_text)

        # Averages
        avg_val_loss = val_loss / len(val_loader)
        val_perplexity = calculate_perplexity(avg_val_loss)
        avg_val_bleu = val_bleu / len(val_loader.dataset)
        avg_val_cer = val_cer / len(val_loader.dataset)
        avg_val_edit = val_edit / len(val_loader.dataset)
        val_accuracy = val_correct / val_total

        # ---- RESULTS ----
        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f}, Perplexity: {train_perplexity:.4f}, Accuracy: {train_accuracy:.4f}")
        print(f"Val Loss:   {avg_val_loss:.4f}, Perplexity: {val_perplexity:.4f}, "
              f"Accuracy: {val_accuracy:.4f}, BLEU: {avg_val_bleu:.4f}, CER: {avg_val_cer:.4f}, "
              f"Edit Dist: {avg_val_edit:.2f}")




def evaluate_model(model, test_loader, roman_vocab, device):
    """Evaluate model with metrics"""
    model.eval()
    id2roman = {v: k for k, v in roman_vocab.items()}

    total_bleu = 0
    total_cer = 0
    total_edit_dist = 0
    total_loss = 0
    count = 0

    criterion = nn.CrossEntropyLoss(ignore_index=0)

    with torch.no_grad():
        for src, tgt in test_loader:
            src, tgt = src.to(device), tgt.to(device)

            output = model(src)

            # Calculate loss
            loss_output_flat = output.reshape(-1, output.size(-1))
            target_flat = tgt.reshape(-1)
            loss = criterion(loss_output_flat, target_flat)
            total_loss += loss.item()

            predictions = torch.argmax(output, dim=-1)

            for i in range(src.size(0)):
                pred_tokens = predictions[i].cpu().numpy()
                tgt_tokens = tgt[i].cpu().numpy()

                pred_text = decode_tokens(pred_tokens, id2roman)
                ref_text = decode_tokens(tgt_tokens, id2roman)

                total_bleu += calculate_bleu(ref_text, pred_text)
                total_cer += calculate_cer(ref_text, pred_text)
                total_edit_dist += calculate_edit_distance(ref_text, pred_text)
                count += 1

    avg_loss = total_loss / len(test_loader)
    avg_bleu = total_bleu / count
    avg_cer = total_cer / count
    avg_edit_dist = total_edit_dist / count

    print(f"Test Loss: {avg_loss:.4f}, Perplexity: {calculate_perplexity(avg_loss):.4f}")
    print(f"BLEU Score: {avg_bleu:.4f}")
    print(f"Character Error Rate: {avg_cer:.4f}")
    print(f"Average Edit Distance: {avg_edit_dist:.2f}")

    return avg_bleu, avg_cer, avg_edit_dist

def show_examples(model, test_dataset, urdu_vocab, roman_vocab, device, num_examples=5):
    """Show translation examples"""
    model.eval()
    id2roman = {v: k for k, v in roman_vocab.items()}
    id2urdu = {v: k for k, v in urdu_vocab.items()}

    indices = random.sample(range(len(test_dataset)), num_examples)

    with torch.no_grad():
        for idx in indices:
            src, tgt = test_dataset[idx]
            src_batch = src.unsqueeze(0).to(device)

            output = model(src_batch)
            prediction = torch.argmax(output, dim=-1).squeeze(0)

            src_text = decode_tokens(src.numpy(), id2urdu)
            tgt_text = decode_tokens(tgt.numpy(), id2roman)
            pred_text = decode_tokens(prediction.cpu().numpy(), id2roman)

            print(f"Source (Urdu): {src_text}")
            print(f"Target (Roman): {tgt_text}")
            print(f"Prediction: {pred_text}")

            # Calculate metrics for this example
            bleu = calculate_bleu(tgt_text, pred_text)
            cer = calculate_cer(tgt_text, pred_text)
            edit_dist = calculate_edit_distance(tgt_text, pred_text)

            print(f"BLEU: {bleu:.3f}, CER: {cer:.3f}, Edit Dist: {edit_dist}")
            print("-" * 50)


In [7]:
def test_metrics_sanity_check():
    """Sanity check for evaluation metrics"""
    print("Testing evaluation metrics...")

    # Test BLEU
    ref = "yeh ek test sentence hai"
    hyp = "yeh ek test sentence hai"
    print(f"BLEU (identical): {calculate_bleu(ref, hyp):.4f}")

    hyp = "yeh test sentence hai"
    print(f"BLEU (missing word): {calculate_bleu(ref, hyp):.4f}")

    # Test CER
    ref = "hello world"
    hyp = "hello world"
    print(f"CER (identical): {calculate_cer(ref, hyp):.4f}")

    hyp = "helo wrold"
    print(f"CER (2 errors): {calculate_cer(ref, hyp):.4f}")

    # Test Edit Distance
    print(f"Edit distance (identical): {calculate_edit_distance('hello', 'hello')}")
    print(f"Edit distance (1 substitution): {calculate_edit_distance('hello', 'hallo')}")


In [8]:
test_metrics_sanity_check()

Testing evaluation metrics...
BLEU (identical): 1.1167
BLEU (missing word): 0.3864
CER (identical): 0.0000
CER (2 errors): 0.2727
Edit distance (identical): 0
Edit distance (1 substitution): 1


In [9]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [10]:
# Load data
src_sentences, tgt_sentences, urdu_vocab, roman_vocab = load_data_and_vocab()


In [11]:
# Create datasets
train_dataset, val_dataset, test_dataset = create_datasets(
    src_sentences, tgt_sentences, urdu_vocab, roman_vocab)


In [12]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)


In [13]:

# Initialize model
model = Seq2SeqModel().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding


# Print detailed model parameter sizes
encoder_size = sum(p.numel() for p in model.encoder.parameters())
decoder_size = sum(p.numel() for p in model.decoder.parameters())
total_size = encoder_size + decoder_size

print(f"Encoder size (number of parameters): {encoder_size}")
print(f"Decoder size (number of parameters): {decoder_size}")
print(f"Total model size (encoder + decoder): {total_size}")

# Print model size (number of parameters)
model_size = sum(p.numel() for p in model.parameters())
print(f"Model size (number of parameters): {model_size}")


Encoder size (number of parameters): 724992
Decoder size (number of parameters): 2499584
Total model size (encoder + decoder): 3224576
Model size (number of parameters): 3224576


In [14]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _ترے _ب غ یر _کبھی _گھر _میں _رو شن ی _نہ _ہوئی
Target (Roman): tire_ ba ghair_ ka bhi_ gha r_ men_ rau sh ni _ na_ hui_
Prediction: ik_ aise_ aise_ aise_ aise_ bi aise_ aise_ aise_ aise_ aise_ aise_ aise_ aise_
BLEU: 0.000, CER: 0.946, Edit Dist: 53
--------------------------------------------------
Source (Urdu): _کی ا _ستم _ہے _کہ _ہم _لوگ _مر _جائیں _گے
Target (Roman): kya_ sitam_ hai_ ki_ ham_ log_ ma r_ ja enge_
Prediction: chhod_ ve_ ve_ ve_ bi bi bi bi bi bi bi bi
BLEU: 0.000, CER: 0.778, Edit Dist: 35
--------------------------------------------------
Source (Urdu): _ح یر تی _ہے _یہ _آئی نہ _کس _کا
Target (Roman): ha ir a ti_ hai_ ye_ a in a_ ki s_ ka_
Prediction: na_ aise_ aise_ bi bi bi aise_ aise_ aise_ aise_ aise_
BLEU: 0.000, CER: 0.868, Edit Dist: 33
--------------------------------------------------
Source (Urdu): _جا ن _پی ار ی _بھی _نہیں _جا ن _سے _جا تے _بھی _نہیں
Target (Roman): ja an _ p ya ri_ bhi_ nahin_ ja an _ se_ jaate_ bh

In [15]:
print("Starting training...")

# Step 1: Train encoder only (freeze decoder)
print("\nStep 1: Training encoder (5 epochs)")
for param in model.decoder.parameters():
    param.requires_grad = False

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)

train_model(model, train_loader, val_loader, criterion, optimizer, 10, device, roman_vocab)


Starting training...

Step 1: Training encoder (5 epochs)

Epoch 1/10
Train Loss: 6.0404, Perplexity: 420.0544, Accuracy: 0.0368
Val Loss:   6.0281, Perplexity: 414.9054, Accuracy: 0.0378, BLEU: 0.0033, CER: 1.0960, Edit Dist: 58.76

Epoch 2/10
Train Loss: 6.0225, Perplexity: 412.6065, Accuracy: 0.0373
Val Loss:   6.0180, Perplexity: 410.7458, Accuracy: 0.0376, BLEU: 0.0032, CER: 1.0718, Edit Dist: 57.53

Epoch 3/10
Train Loss: 6.0138, Perplexity: 409.0538, Accuracy: 0.0370
Val Loss:   6.0118, Perplexity: 408.2114, Accuracy: 0.0376, BLEU: 0.0030, CER: 1.0948, Edit Dist: 58.74

Epoch 4/10
Train Loss: 6.0086, Perplexity: 406.9211, Accuracy: 0.0370
Val Loss:   6.0076, Perplexity: 406.4938, Accuracy: 0.0376, BLEU: 0.0031, CER: 1.0971, Edit Dist: 58.85

Epoch 5/10
Train Loss: 6.0047, Perplexity: 405.3316, Accuracy: 0.0369
Val Loss:   6.0043, Perplexity: 405.1594, Accuracy: 0.0377, BLEU: 0.0032, CER: 1.1112, Edit Dist: 59.60

Epoch 6/10
Train Loss: 6.0018, Perplexity: 404.1665, Accuracy: 0.0

In [16]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _بس _چ پ _رہ و _ہم ار ے _بھی _من ہ _میں _ز ب ان _ہے
Target (Roman): ba s_ ch u p_ ra ho_ ham ar e_ bhi_ munh_ men_ za ba n_ hai_
Prediction: has bi bi bi aise_ aise_ aise_ aise_ aise_ aise_
BLEU: 0.000, CER: 0.667, Edit Dist: 40
--------------------------------------------------
Source (Urdu): _در _ی ار _ک ع ب ہ _بن تا _جو _مر ا _م زار _ہوتا
Target (Roman): da r-e- ya r_ ka ab a_ ba n ta _ jo_ mi ra _ ma za r_ hota_
Prediction: kare_ aise_ aise_ aise_ bi bi bi bi bi
BLEU: 0.000, CER: 0.712, Edit Dist: 42
--------------------------------------------------
Source (Urdu): _میں _کوئی _غیر _نہیں _ہو ں _کہ _چ ھ پا ؤ _ی ار و
Target (Roman): main_ koi_ ghair_ nahin_ huun_ ki_ chhu pa o_ ya ar o_
Prediction: kare_ aise_ bi bi bi bi bi bi
BLEU: 0.000, CER: 0.741, Edit Dist: 40
--------------------------------------------------
Source (Urdu): _اس _مل ک _کا _ہر _خط ہ _ت ات ار _نظر _آیا
Target (Roman): is _ mu l k_ ka_ ha r_ kh it ta _ ta ta r_ nazar_ aaya_
Pre

In [17]:
# Step 2: Train decoder only (freeze encoder)
print("\nStep 2: Training decoder (10 epochs)")
for param in model.encoder.parameters():
    param.requires_grad = False
for param in model.decoder.parameters():
    param.requires_grad = True

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)
train_model(model, train_loader, val_loader, criterion, optimizer, 15, device,roman_vocab)



Step 2: Training decoder (10 epochs)

Epoch 1/15
Train Loss: 4.6376, Perplexity: 103.2922, Accuracy: 0.1123
Val Loss:   4.4075, Perplexity: 82.0617, Accuracy: 0.1264, BLEU: 0.0714, CER: 0.5889, Edit Dist: 33.10

Epoch 2/15
Train Loss: 4.2077, Perplexity: 67.2015, Accuracy: 0.1376
Val Loss:   4.0199, Perplexity: 55.6942, Accuracy: 0.1548, BLEU: 0.0984, CER: 0.5201, Edit Dist: 29.20

Epoch 3/15
Train Loss: 3.8611, Perplexity: 47.5196, Accuracy: 0.1651
Val Loss:   3.6901, Perplexity: 40.0503, Accuracy: 0.1831, BLEU: 0.1188, CER: 0.4760, Edit Dist: 26.77

Epoch 4/15
Train Loss: 3.5604, Perplexity: 35.1756, Accuracy: 0.1870
Val Loss:   3.4117, Perplexity: 30.3158, Accuracy: 0.2050, BLEU: 0.1415, CER: 0.4364, Edit Dist: 24.55

Epoch 5/15
Train Loss: 3.2887, Perplexity: 26.8080, Accuracy: 0.2094
Val Loss:   3.1139, Perplexity: 22.5087, Accuracy: 0.2320, BLEU: 0.1709, CER: 0.4089, Edit Dist: 23.04

Epoch 6/15
Train Loss: 3.0533, Perplexity: 21.1859, Accuracy: 0.2296
Val Loss:   2.9006, Perple

In [18]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _ہو _ف شا ر _ض ع ف _میں _کی ا _ن ات وا نی _کی _نم ود
Target (Roman): ho_ fi sh ar -e- z o f_ men_ kya_ na- ta va ni _ ki_ nu mud _
Prediction: ho_ fa sh ar -e- -e- af f_ men_ kya_ ne_ ta va ni _ ki_ na nu
BLEU: 0.411, CER: 0.213, Edit Dist: 13
--------------------------------------------------
Source (Urdu): _پا ب ند ی وں _ن ے _عشق _کی _بی ک س _رکھ ا _مجھ ے
Target (Roman): pa ba n di yon_ ne_ ishq_ ki_ be ka s_ ra kh a_ mujhe_
Prediction: pa ab n da yon_ ne_ ne_ ne_ ki_ ya k_ _ ra kh a_ mujhe_
BLEU: 0.282, CER: 0.241, Edit Dist: 13
--------------------------------------------------
Source (Urdu): _اک _پ ل _کے _ج ھ پ ک نے _تک _ہر _ک ھی ل _سہ ان ا _ہے
Target (Roman): ik_ pa l_ ke_ j ha pa k ne_ tak_ ha r_ khe l_ su ha na_ hai_
Prediction: ik_ pa l_ ke_ j h_ pa k_ ne_ tak_ r_ r_ in l_ sa ha na_ hai_
BLEU: 0.405, CER: 0.133, Edit Dist: 8
--------------------------------------------------
Source (Urdu): _بھ ڑ ک ے _ہے _دل _کی _آ ت ش _تجھ _ن ی ہ _کی _ہوا 

In [19]:
#more 10 epochs
# Step 2: Train decoder only (freeze encoder)
print("\nStep 2: Training decoder (10 epochs)")
for param in model.encoder.parameters():
    param.requires_grad = False
for param in model.decoder.parameters():
    param.requires_grad = True

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)
train_model(model, train_loader, val_loader, criterion, optimizer, 10, device,roman_vocab)



Step 2: Training decoder (10 epochs)

Epoch 1/10
Train Loss: 2.0038, Perplexity: 7.4173, Accuracy: 0.3436
Val Loss:   2.1335, Perplexity: 8.4442, Accuracy: 0.3429, BLEU: 0.3893, CER: 0.2683, Edit Dist: 15.17

Epoch 2/10
Train Loss: 1.9501, Perplexity: 7.0295, Accuracy: 0.3487
Val Loss:   2.1386, Perplexity: 8.4871, Accuracy: 0.3463, BLEU: 0.4064, CER: 0.2556, Edit Dist: 14.45

Epoch 3/10
Train Loss: 1.9111, Perplexity: 6.7603, Accuracy: 0.3538
Val Loss:   2.0917, Perplexity: 8.0983, Accuracy: 0.3514, BLEU: 0.4094, CER: 0.2569, Edit Dist: 14.52

Epoch 4/10
Train Loss: 1.8614, Perplexity: 6.4329, Accuracy: 0.3613
Val Loss:   2.0760, Perplexity: 7.9725, Accuracy: 0.3534, BLEU: 0.4241, CER: 0.2485, Edit Dist: 14.08

Epoch 5/10
Train Loss: 1.8194, Perplexity: 6.1684, Accuracy: 0.3655
Val Loss:   2.0645, Perplexity: 7.8816, Accuracy: 0.3579, BLEU: 0.4371, CER: 0.2456, Edit Dist: 13.89

Epoch 6/10
Train Loss: 1.7868, Perplexity: 5.9701, Accuracy: 0.3717
Val Loss:   2.0603, Perplexity: 7.8480

In [20]:
# Step 3: Train full model
print("\nStep 3: Training full model (10 epochs)")
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=0.0005)
train_model(model, train_loader, val_loader, criterion, optimizer, 20, device,roman_vocab)



Step 3: Training full model (10 epochs)

Epoch 1/20
Train Loss: 1.5643, Perplexity: 4.7791, Accuracy: 0.4031
Val Loss:   1.9829, Perplexity: 7.2636, Accuracy: 0.3768, BLEU: 0.4826, CER: 0.2209, Edit Dist: 12.46

Epoch 2/20
Train Loss: 1.5154, Perplexity: 4.5513, Accuracy: 0.4101
Val Loss:   1.9793, Perplexity: 7.2378, Accuracy: 0.3782, BLEU: 0.4892, CER: 0.2203, Edit Dist: 12.45

Epoch 3/20
Train Loss: 1.4808, Perplexity: 4.3966, Accuracy: 0.4157
Val Loss:   1.9848, Perplexity: 7.2779, Accuracy: 0.3799, BLEU: 0.4832, CER: 0.2224, Edit Dist: 12.56

Epoch 4/20
Train Loss: 1.4570, Perplexity: 4.2929, Accuracy: 0.4195
Val Loss:   1.9996, Perplexity: 7.3865, Accuracy: 0.3804, BLEU: 0.4925, CER: 0.2187, Edit Dist: 12.34

Epoch 5/20
Train Loss: 1.4262, Perplexity: 4.1627, Accuracy: 0.4227
Val Loss:   1.9619, Perplexity: 7.1127, Accuracy: 0.3843, BLEU: 0.4981, CER: 0.2140, Edit Dist: 12.10

Epoch 6/20
Train Loss: 1.3974, Perplexity: 4.0445, Accuracy: 0.4264
Val Loss:   1.9728, Perplexity: 7.1

In [21]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _ہر _ایک _سم ت _سے _اک _آ فت اب _اب ھر ے _گا
Target (Roman): ha r_ ek_ sam t_ se_ ik_ af ta b_ u b h re ga _
Prediction: ha r_ ek_ sam t_ se_ ik_ af ta b_ u bha r ega_
BLEU: 0.752, CER: 0.106, Edit Dist: 5
--------------------------------------------------
Source (Urdu): _ض د _ہے _ا نہ یں _پو را _مر ا _ا رم اں _نہ _کر یں _گے
Target (Roman): zi d_ hai_ un hen_ pu u ra _ mi ra _ ar man_ na_ ka re nge_
Prediction: z d_ hai_ ul hen_ qa u ra _ mi ra _ ar man_ ka ka re nge_
BLEU: 0.601, CER: 0.102, Edit Dist: 6
--------------------------------------------------
Source (Urdu): _مجھ ے _میرے _سو ا _سب _لوگ _سمجھ یں
Target (Roman): mujhe_ mere_ siva_ sa b_ log_ samj hen_
Prediction: mujhe_ di ju ju ja b_ log_ samj hen_
BLEU: 0.380, CER: 0.282, Edit Dist: 11
--------------------------------------------------
Source (Urdu): _کرو ں _کی ا _یہ _بھی _تو _ن ا _ط ا قت ی _سے _ہو _نہیں _سک تا
Target (Roman): ka ru n_ kya_ ye_ bhi_ to_ na- ta qat i_ se_ ho_ nahin_ sak 

In [22]:
# Step 4: Fine-tune with low learning rate and decay
print("\nStep 4: Fine-tuning with learning rate decay (10 epochs)")
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

for epoch in range(10):
    train_model(model, train_loader, val_loader, criterion, optimizer, 1, device, roman_vocab)
    scheduler.step()
    print(f"Learning rate: {scheduler.get_last_lr()[0]:.6f}")



Step 4: Fine-tuning with learning rate decay (10 epochs)

Epoch 1/1
Train Loss: 1.1062, Perplexity: 3.0229, Accuracy: 0.4687
Val Loss:   2.0053, Perplexity: 7.4286, Accuracy: 0.3970, BLEU: 0.5332, CER: 0.1983, Edit Dist: 11.16
Learning rate: 0.000090

Epoch 1/1
Train Loss: 1.0811, Perplexity: 2.9480, Accuracy: 0.4726
Val Loss:   2.0135, Perplexity: 7.4892, Accuracy: 0.3971, BLEU: 0.5360, CER: 0.1968, Edit Dist: 11.06
Learning rate: 0.000081

Epoch 1/1
Train Loss: 1.0731, Perplexity: 2.9244, Accuracy: 0.4754
Val Loss:   2.0059, Perplexity: 7.4330, Accuracy: 0.3977, BLEU: 0.5348, CER: 0.1968, Edit Dist: 11.06
Learning rate: 0.000073

Epoch 1/1
Train Loss: 1.0614, Perplexity: 2.8905, Accuracy: 0.4763
Val Loss:   2.0132, Perplexity: 7.4869, Accuracy: 0.3979, BLEU: 0.5361, CER: 0.1964, Edit Dist: 11.05
Learning rate: 0.000066

Epoch 1/1
Train Loss: 1.0560, Perplexity: 2.8748, Accuracy: 0.4765
Val Loss:   2.0186, Perplexity: 7.5276, Accuracy: 0.3980, BLEU: 0.5348, CER: 0.1977, Edit Dist: 11

In [23]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _اپنے _جن گ ل _سے _جو _گھ بر ا _کے _ا ڑ ے _تھے _پی ا س ے
Target (Roman): apne_ ja ng a l_ se_ jo_ gha b ra _ ke_ u de _ the_ p ya se_
Prediction: apne_ ja ng a l_ se_ jo_ gha b ra _ ke_ u de _ _ p ya e_
BLEU: 0.917, CER: 0.067, Edit Dist: 4
--------------------------------------------------
Source (Urdu): _میں _ن ے _خود _سے _ن ب اہ _کر _ل ی _ہے
Target (Roman): main_ ne_ khud_ se_ ni ba h_ ka r_ li _ hai_
Prediction: main_ ne_ khud_ se_ se_ ne_ _ tha_ r_ la _ hai_
BLEU: 0.336, CER: 0.250, Edit Dist: 11
--------------------------------------------------
Source (Urdu): _جو _پر _غ ر ور _ک ھ نچ تا _ہے _م اہ _م بی ں _سے _دو ر
Target (Roman): jo_ pu r- gh u ru r_ kh in ch ta _ hai_ ma h -e- mu bi n_ se_ duur_
Prediction: jo_ pa r_ gh u ru r_ kh in ch ta _ hai_ ma h _ mi ji n_
BLEU: 0.653, CER: 0.254, Edit Dist: 17
--------------------------------------------------
Source (Urdu): _ر ات _ب ج تی _تھی _دو ر _ش ہ نا ئی
Target (Roman): raat_ ba j ti_ thi_ duur_

In [24]:
# Evaluation using method 5
print("\nEvaluation Results:")
evaluate_model(model, test_loader, roman_vocab, device)

# Evaluation Results:
# Test Loss: 2.0089, Perplexity: 7.4549
# BLEU Score: 0.4518
# Character Error Rate: 0.1949
# Average Edit Distance: 10.96
# (0.45178210247696443, 0.19485231869574338, 10.957174816235218)


Evaluation Results:
Test Loss: 1.9561, Perplexity: 7.0719
BLEU Score: 0.5293
Character Error Rate: 0.1977
Average Edit Distance: 11.03


(0.5292834548778224, 0.19770934807207083, 11.029402364972835)

In [27]:
def calculate_bleu(reference, hypothesis):
    """Calculate BLEU score"""
    reference_tokens = reference.split()
    hypothesis_tokens = hypothesis.split()

    if len(hypothesis_tokens) == 0:
        return 0.0

    smoothie = SmoothingFunction().method0
    return sentence_bleu([reference_tokens], hypothesis_tokens, smoothing_function=smoothie)


# Evaluation using method 0
print("\nEvaluation Results:")
evaluate_model(model, test_loader, roman_vocab, device)




Evaluation Results:


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

Test Loss: 1.9561, Perplexity: 7.0719
BLEU Score: 0.4234
Character Error Rate: 0.1977
Average Edit Distance: 11.03


(0.423390754068661, 0.19770934807207083, 11.029402364972835)

In [28]:
def calculate_bleu(reference, hypothesis):
    """Calculate BLEU score"""
    reference_tokens = reference.split()
    hypothesis_tokens = hypothesis.split()

    if len(hypothesis_tokens) == 0:
        return 0.0

    smoothie = SmoothingFunction().method1
    return sentence_bleu([reference_tokens], hypothesis_tokens, smoothing_function=smoothie)


# Evaluation using method 1
print("\nEvaluation Results:")
evaluate_model(model, test_loader, roman_vocab, device)




Evaluation Results:
Test Loss: 1.9561, Perplexity: 7.0719
BLEU Score: 0.4380
Character Error Rate: 0.1977
Average Edit Distance: 11.03


(0.43802178398695313, 0.19770934807207083, 11.029402364972835)

In [25]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _ق یا م ت _ہے _کہ _ہو وے _مد ع ی _کا _ہم _سفر _غالبؔ
Target (Roman): qayamat_ hai_ ki_ ho ve_ mud da i_ ka_ ham- sa fa r_ 'ghalib'_
Prediction: qayamat_ hai_ hai_ hai_ qayamat_ _ _ _ _ hai_ r_ r_ _ r_ 'ghalib'_
BLEU: 0.128, CER: 0.403, Edit Dist: 25
--------------------------------------------------
Source (Urdu): _میں _تجھ _سے _ملتا _تو _ت ف ص ی ل _میں _نہیں _جا تا
Target (Roman): main_ tujh_ se_ milta_ to_ ta f si l_ men_ nahin_ jaata_
Prediction: main_ tujh_ se_ milta_ to_ ta f s _ men_ sh sak an
BLEU: 0.581, CER: 0.196, Edit Dist: 11
--------------------------------------------------
Source (Urdu): _ق ص ر _ٹ و ٹ ے _نہ _بے _ن وا ئی _گئی
Target (Roman): qa s r_ tu ut e_ na_ be- na va i_ ga i_
Prediction: qa s r_ tu ut e_ ga ta s a r_ ki_ i_ i_
BLEU: 0.477, CER: 0.308, Edit Dist: 12
--------------------------------------------------
Source (Urdu): _ک ٹ _جائیں _میری _سوچ _کے _پر _تم _کو _اس _سے _کی ا
Target (Roman): ka t_ jaaen_ meri_ so ch _ ke_ p

In [26]:
save_path = "/content/drive/MyDrive/Model2/urdu_roman_nmt_model.pth"
# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'urdu_vocab': urdu_vocab,
    'roman_vocab': roman_vocab
}, save_path)
print("Model saved as 'urdu_roman_nmt_model.pth'")



Model saved as 'urdu_roman_nmt_model.pth'
